# Лабораторная работа №3. Вариант 4.2: NYC Airbnb Open Data

**Уровень 4 (продвинутый) — 8–8.5 баллов**

**Датасет:** NYC Airbnb Open Data (Kaggle: dgomonov/nyc-airbnb)  
**Размер:** 49K × 16  
**Целевая переменная:** `price`  
**Категориальные:** `neighbourhood_group`, `room_type`, `neighbourhood`  
**Текстовые:** `name`, `host_name`  

**Специфика варианта:**  
- Geographic FE: lat/lon → кластеры через KMeans  
- Target encoding для `neighbourhood`  
- SHAP для `room_type`  


## 1. Импорты и проверка библиотек

Устанавливаем и импортируем все необходимые библиотеки.  
Проверяем, что каждая из них доступна — если нет, выводим предупреждение.

**Библиотеки:**
- `polars` — основная для работы с данными (eager + lazy API)
- `pandas` — для benchmark-сравнения и совместимости с sklearn
- `numpy`, `matplotlib`, `seaborn` — вычисления и визуализация
- `sklearn` — модели, метрики, отбор признаков, кросс-валидация
- `xgboost` — градиентный бустинг
- `optuna` — оптимизация гиперпараметров
- `shap` — интерпретация моделей
- `duckdb` — SQL поверх Polars DataFrame
- `scipy` — статистические тесты


In [ ]:
pip install optuna

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

import xgboost as xgb

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap

import duckdb

# Статистика
from scipy import stats
from scipy.stats import shapiro
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan

print("Все библиотеки импортированы успешно!")
print(f"Polars version: {pl.__version__}")
print(f"Optuna version: {optuna.__version__}")


 # !!! Генерация исключительно для примера, в лабораторной берем данные из варианта в задании

## 2. Генерация синтетических данных NYC Airbnb


Так как ноутбук запускается в изолированном контейнере без доступа к интернету,  
генерируем синтетические данные, структурно совпадающие с реальным датасетом NYC Airbnb:

**Колонки (16):**
- `id` — уникальный идентификатор объявления
- `name` — название объявления (текст)
- `host_id` — идентификатор хоста
- `host_name` — имя хоста (текст)
- `neighbourhood_group` — район NYC (Manhattan, Brooklyn, Queens, Bronx, Staten Island)
- `neighbourhood` — микрорайон (~220 уникальных)
- `latitude`, `longitude` — координаты
- `room_type` — тип комнаты (Entire home/apt, Private room, Shared room)
- `price` — цена за ночь (целевая переменная)
- `minimum_nights` — минимум ночей
- `number_of_reviews` — количество отзывов
- `last_review` — дата последнего отзыва
- `reviews_per_month` — отзывов в месяц
- `calculated_host_listings_count` — количество объявлений хоста
- `availability_365` — доступность в году

**Логика генерации цены:**  
- Manhattan → дороже (база 150 + шум)  
- Brooklyn → средне (база 90 + шум)  
- Queens → дешевле (база 70 + шум)  
- Bronx, Staten Island → самые дешёвые (база 50 + шум)  
- Entire home/apt → ×1.5, Private room → ×1.0, Shared room → ×0.6  
- Больше отзывов → небольшой дисконт


In [ ]:
# Генерация синтетических данных NYC Airbnb
np.random.seed(42)
N = 49000  # 49K строк, как в реальном датасете

# Районы NYC с вероятностями (Manhattan и Brooklyn — самые популярные)
neighbourhood_groups = np.random.choice(
    ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'],
    size=N, p=[0.44, 0.41, 0.10, 0.03, 0.02]
)

# Микрорайоны (упрощённый набор)
neighbourhood_map = {
    'Manhattan': ['Harlem', 'Upper East Side', 'Midtown', 'Lower East Side', 'Chelsea', 'SoHo', 'Times Square', 'Greenwich Village'],
    'Brooklyn': ['Williamsburg', 'Bedford-Stuyvesant', 'Bushwick', 'Crown Heights', 'Park Slope', 'DUMBO', 'Greenpoint'],
    'Queens': ['Astoria', 'Long Island City', 'Flushing', 'Jamaica', 'Ridgewood'],
    'Bronx': ['Mott Haven', 'Fordham', 'Riverdale', 'Pelham Bay'],
    'Staten Island': ['St. George', 'Tottenville', 'Stapleton']
}
neighbourhoods = [np.random.choice(neighbourhood_map[g]) for g in neighbourhood_groups]

# Типы комнат
room_types = np.random.choice(
    ['Entire home/apt', 'Private room', 'Shared room'],
    size=N, p=[0.52, 0.43, 0.05]
)

# Координаты (приблизительные для NYC)
lat_base = {'Manhattan': 40.78, 'Brooklyn': 40.65, 'Queens': 40.73, 'Bronx': 40.84, 'Staten Island': 40.58}
lon_base = {'Manhattan': -73.97, 'Brooklyn': -73.95, 'Queens': -73.80, 'Bronx': -73.87, 'Staten Island': -74.15}
latitudes = [lat_base[g] + np.random.normal(0, 0.03) for g in neighbourhood_groups]
longitudes = [lon_base[g] + np.random.normal(0, 0.03) for g in neighbourhood_groups]

# Базовая цена по районам
price_base = {'Manhattan': 150, 'Brooklyn': 90, 'Queens': 70, 'Bronx': 50, 'Staten Island': 50}
room_mult = {'Entire home/apt': 1.5, 'Private room': 1.0, 'Shared room': 0.6}

# Генерация цены
prices = []
for i in range(N):
    base = price_base[neighbourhood_groups[i]]
    mult = room_mult[room_types[i]]
    noise = np.random.exponential(30)  # экспоненциальный шум → скошенное распределение
    p = base * mult + noise
    # Дисконт за популярность (много отзывов → дешевле)
    reviews = np.random.poisson(20)
    if reviews > 50:
        p *= 0.85
    prices.append(max(10, round(p)))  # минимальная цена $10

# Остальные колонки
minimum_nights = np.random.choice([1, 2, 3, 5, 7, 14, 30], size=N, p=[0.5, 0.15, 0.1, 0.1, 0.08, 0.04, 0.03])
number_of_reviews = np.random.poisson(25, size=N)
reviews_per_month = number_of_reviews / 12 + np.random.normal(0, 0.3, size=N)
reviews_per_month = np.clip(reviews_per_month, 0, None)  # не может быть отрицательным
host_listings = np.random.choice([1, 1, 1, 2, 2, 3, 5, 10, 20, 50], size=N)
availability = np.random.randint(0, 365, size=N)

# Имена хостов
first_names = ['John', 'Sarah', 'Michael', 'Emma', 'David', 'Lisa', 'James', 'Anna', 'Robert', 'Maria']
last_names = ['Smith', 'Johnson', 'Brown', 'Davis', 'Wilson', 'Taylor', 'Anderson', 'Thomas', 'Lee', 'Walker']
host_names = [f"{np.random.choice(first_names)} {np.random.choice(last_names)}" for _ in range(N)]
host_ids = np.random.randint(1000, 500000, size=N)

# Названия объявлений
name_templates = [
    'Cozy {room} in {nb}', 'Beautiful {room} near {nb}', 'Spacious apartment in {nb}',
    'Luxury {room} | {nb}', 'Stylish loft in {nb}', 'Modern {room} in heart of {nb}',
    'Charming {room} in {nb}', 'Sunny {room} in {nb}', 'Perfect {room} in {nb}'
]
listing_names = [np.random.choice(name_templates).format(
    room='studio' if room_types[i] == 'Entire home/apt' else 'room',
    nb=neighbourhoods[i]
) for i in range(N)]

# Даты последних отзывов (некоторые пропущены → null)
last_review_dates = pd.date_range('2019-01-01', '2019-07-01', periods=N)
has_review = np.random.random(N) > 0.15  # ~15% без отзывов
last_review = [d.strftime('%Y-%m-%d') if has_review[i] else None for i, d in enumerate(last_review_dates)]

# Сборка в Polars DataFrame
df = pl.DataFrame({
    'id': np.arange(1, N + 1),
    'name': listing_names,
    'host_id': host_ids,
    'host_name': host_names,
    'neighbourhood_group': neighbourhood_groups,
    'neighbourhood': neighbourhoods,
    'latitude': latitudes,
    'longitude': longitudes,
    'room_type': room_types,
    'price': prices,
    'minimum_nights': minimum_nights,
    'number_of_reviews': number_of_reviews,
    'last_review': last_review,
    'reviews_per_month': reviews_per_month,
    'calculated_host_listings_count': host_listings,
    'availability_365': availability
})

# Добавляем пропуски в reviews_per_month там, где нет отзывов
df = df.with_columns(
    pl.when(pl.col('last_review').is_null())
      .then(None)
      .otherwise(pl.col('reviews_per_month'))
      .alias('reviews_per_month')
)

print(f"Данные сгенерированы: {df.shape[0]} строк × {df.shape[1]} колонок")


## 3. Загрузка данных и первые 10 строк

В реальном проекте данные загружаются из CSV:
```python
df = pl.read_csv("AB_NYC_2019.csv")
```

Здесь мы используем уже сгенерированный DataFrame.  
Выводим первые 10 строк методом `.head(10)`.


In [ ]:
print("Первые 10 строк датасета:")
df.head(10)


## 4. Размерность данных

Свойство `.shape` возвращает кортеж `(rows, columns)`.  
В Polars также доступен `.height` и `.width` по отдельности.


In [ ]:
n_rows, n_cols = df.shape
print(f"Размерность данных: {n_rows} строк × {n_cols} столбцов")
print(f"Высота (height): {df.height}")
print(f"Ширина (width): {df.width}")


## 5. Пропущенные значения

Метод `.null_count()` возвращает DataFrame с количеством null в каждой колонке.  
В датасете NYC Airbnb пропуски обычно встречаются в:
- `last_review` — если нет отзывов
- `reviews_per_month` — если нет отзывов
- `name` — редко
- `host_name` — очень редко


In [ ]:
#  Общий пункт 3: Подсчёт пропусков
null_counts = df.null_count()
print("Количество пропущенных значений в каждой переменной:")
print(null_counts)
print()
print("Процент пропусков:")
for col in df.columns:
    n_null = df[col].null_count()
    pct = n_null / df.height * 100
    if n_null > 0:
        print(f"  {col}: {n_null} ({pct:.1f}%)")


## 6. Общий пункт 4: Типы данных и конвертация

Свойство `.schema` показывает тип каждой колонки.  
**Оптимизация:**
- Числа с плавающей точкой → `Float32` (экономия 50% памяти)
- Категориальные строки → `Categorical` (экономия памяти + ускорение group_by)
- Целые числа → `Int32` если значения помещаются

**Типы Polars:**
- `Int64`, `Int32`, `Float64`, `Float32` — числовые
- `Utf8` — строки
- `Categorical` — категориальные (факторизованные строки)
- `Null` — колонка только с null


In [ ]:
print("Исходная схема:")
for col, dtype in df.schema.items():
    print(f"  {col}: {dtype}")

# Конвертация:
# 1. Float64 → Float32 для экономии памяти
# 2. Строковые категориальные → Categorical
# 3. Целые числа → Int32
df = df.with_columns([
    pl.col('latitude').cast(pl.Float32),
    pl.col('longitude').cast(pl.Float32),
    pl.col('price').cast(pl.Int32),
    pl.col('minimum_nights').cast(pl.Int32),
    pl.col('number_of_reviews').cast(pl.Int32),
    pl.col('calculated_host_listings_count').cast(pl.Int32),
    pl.col('availability_365').cast(pl.Int32),
    pl.col('reviews_per_month').cast(pl.Float32),
    pl.col('neighbourhood_group').cast(pl.Categorical),
    pl.col('neighbourhood').cast(pl.Categorical),
    pl.col('room_type').cast(pl.Categorical),
])

print("\nПосле конвертации:")
for col, dtype in df.schema.items():
    print(f"  {col}: {dtype}")


## 7. Создание категориальной переменной `price_tier`

Создаём новую категориальную переменную на основе условной логики через `pl.when().then()`.

**Правило:**
- `price < 75` → `"budget"` (бюджетные)
- `75 ≤ price < 200` → `"mid_range"` (средние)
- `price ≥ 200` → `"luxury"` (люкс)

Также удаляем наблюдения без указания цены (если есть null в `price`).  
Эта переменная пригодится позже для сравнения качества модели до и после добавления.


In [ ]:
# Используем pl.when().then() для условной логики
df = df.with_columns(
    pl.when(pl.col('price') < 75).then(pl.lit('budget'))
      .when((pl.col('price') >= 75) & (pl.col('price') < 200)).then(pl.lit('mid_range'))
      .when(pl.col('price') >= 200).then(pl.lit('luxury'))
      .otherwise(None)  # если price = null
      .alias('price_tier')
      .cast(pl.Categorical)
)

# Удаляем строки, где price_tier не указан (price был null)
n_before = df.height
df = df.filter(pl.col('price_tier').is_not_null())
n_after = df.height
print(f"Создана переменная price_tier с категориями: budget, mid_range, luxury")
print(f"Удалено строк без price_tier: {n_before - n_after}")
print(f"Осталось строк: {n_after}")

# Распределение по категориям
print("\nРаспределение price_tier:")
print(df.group_by('price_tier').agg(pl.len().alias('count')).sort('count', descending=True))


## 8. Визуализация распределений

Строим гистограммы и boxplots для ключевых числовых переменных:
- `price` — целевая переменная (ожидается скошенное распределение)
- `minimum_nights` — минимум ночей
- `number_of_reviews` — количество отзывов
- `availability_365` — доступность

Также строим boxplot `price` по `neighbourhood_group` — чтобы увидеть разницу в ценах между районами.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Гистограмма price (с ограничением для читаемости — до $1000)
price_data = df.filter(pl.col('price') < 1000)['price'].to_numpy()
axes[0, 0].hist(price_data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('Распределение price (до $1000)')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Count')

# Гистограмма log(price)
log_price = np.log1p(df['price'].to_numpy())
axes[0, 1].hist(log_price, bins=50, color='seagreen', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('Распределение log(1 + price)')
axes[0, 1].set_xlabel('log(1 + price)')

# Гистограмма minimum_nights (ограничение до 30)
mn_data = df.filter(pl.col('minimum_nights') <= 30)['minimum_nights'].to_numpy()
axes[0, 2].hist(mn_data, bins=30, color='orange', edgecolor='white', alpha=0.8)
axes[0, 2].set_title('Распределение minimum_nights')
axes[0, 2].set_xlabel('Minimum nights')

# Boxplot price по neighbourhood_group
price_by_group = [df.filter(pl.col('neighbourhood_group') == g)['price'].to_numpy() for g in ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']]
axes[1, 0].boxplot(price_by_group, labels=['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Is.'], showfliers=False)
axes[1, 0].set_title('Price по neighbourhood_group')
axes[1, 0].set_ylabel('Price ($)')

# Boxplot price по room_type
price_by_room = [df.filter(pl.col('room_type') == r)['price'].to_numpy() for r in ['Entire home/apt', 'Private room', 'Shared room']]
axes[1, 1].boxplot(price_by_room, labels=['Entire', 'Private', 'Shared'], showfliers=False)
axes[1, 1].set_title('Price по room_type')
axes[1, 1].set_ylabel('Price ($)')

# Гистограмма availability_365
avail_data = df['availability_365'].to_numpy()
axes[1, 2].hist(avail_data, bins=30, color='purple', edgecolor='white', alpha=0.8)
axes[1, 2].set_title('Распределение availability_365')
axes[1, 2].set_xlabel('Days available')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print("Графики сохранены в eda_distributions.png")


## 9. Общий пункт 7: Статистика по категориям

Для ключевой категориальной переменной `neighbourhood_group` рассчитываем:
- Среднюю цену
- Медианную цену
- Количество объявлений

Находим район с **максимальной** и **минимальной** средней ценой.


In [ ]:
# Группируем по neighbourhood_group и считаем статистики
stats_by_group = df.group_by('neighbourhood_group').agg([
    pl.col('price').mean().alias('mean_price'),
    pl.col('price').median().alias('median_price'),
    pl.col('price').min().alias('min_price'),
    pl.col('price').max().alias('max_price'),
    pl.len().alias('count')
]).sort('mean_price', descending=True)

print("Статистика цены по neighbourhood_group:")
print(stats_by_group)

# Находим район с максимальной и минимальной средней ценой
max_group = stats_by_group.row(0, named=True)
min_group = stats_by_group.row(-1, named=True)
print(f"\nРайон с МАКСИМАЛЬНОЙ средней ценой: {max_group['neighbourhood_group']} (${max_group['mean_price']:.2f})")
print(f"Район с МИНИМАЛЬНОЙ средней ценой: {min_group['neighbourhood_group']} (${min_group['mean_price']:.2f})")

# Также статистика по room_type
print("\nСтатистика цены по room_type:")
stats_by_room = df.group_by('room_type').agg([
    pl.col('price').mean().alias('mean_price'),
    pl.col('price').median().alias('median_price'),
    pl.len().alias('count')
]).sort('mean_price', descending=True)
print(stats_by_room)


## 10. Разделение на обучающую и тестовую выборки (80/20)

Используем `train_test_split` из sklearn.  
Перед этим подготавливаем признаки:
- Заполняем пропуски в `reviews_per_month` нулями (нет отзывов → 0)
- Удаляем `last_review` (дата — не числовой признак для базовой модели)
- One-Hot кодируем категориальные переменные


In [ ]:
# Заполняем пропуски в reviews_per_month нулём (нет отзывов = 0 отзывов/мес)
df_clean = df.with_columns(
    pl.col('reviews_per_month').fill_null(0)
)

# Удаляем колонки, которые не нужны для базовой модели
# id, name, host_id, host_name — текстовые/идентификаторы
# last_review — дата (обработаем позже как datetime)
drop_cols = ['id', 'name', 'host_id', 'host_name', 'last_review']
feature_cols = [c for c in df_clean.columns if c not in drop_cols and c != 'price']

# One-Hot кодирование категориальных
categorical_cols = ['neighbourhood_group', 'neighbourhood', 'room_type', 'price_tier']
numerical_cols = [c for c in feature_cols if c not in categorical_cols]

# One-Hot через to_dummies
df_encoded = df_clean.to_dummies(columns=categorical_cols)

# Получаем все колонки признаков (после one-hot)
all_feature_cols = [c for c in df_encoded.columns if c not in drop_cols and c != 'price']

X = df_encoded.select(all_feature_cols).to_numpy()
y = df_encoded['price'].to_numpy().astype(float)

# Разделение 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Обучающая выборка: {X_train.shape[0]} строк, {X_train.shape[1]} признаков")
print(f"Тестовая выборка: {X_test.shape[0]} строк")


## 11. Linear Regression + 4 метрики

Строим базовую модель линейной регрессии.  
Рассчитываем 4 метрики:
- **MSE** (Mean Squared Error) — чувствительна к выбросам
- **RMSE** (Root MSE) — в тех же единицах, что и целевая
- **MAE** (Mean Absolute Error) — устойчива к выбросам
- **R²** (коэффициент детерминации) — доля объяснённой дисперсии

Визуализируем предсказания vs фактические значения.


In [ ]:
# Обучаем модель
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

# 4 метрики
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(" Linear Regression (базовая модель) ")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

# Визуализация: предсказания vs фактические
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.3, s=5, color='steelblue')
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2)
axes[0].set_xlabel('Фактические значения')
axes[0].set_ylabel('Предсказания')
axes[0].set_title(f'Pred vs Actual (R²={r2:.3f})')
axes[0].set_xlim(0, 500)
axes[0].set_ylim(0, 500)

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=5, color='coral')
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Предсказания')
axes[1].set_ylabel('Остатки')
axes[1].set_title('Residuals plot')
axes[1].set_xlim(0, 500)

plt.tight_layout()
plt.savefig('pred_vs_actual.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# simple_feature_cols



```
# Выбран кодовый формат
```

## 12. 5 моделей + bar chart

Строим 5 моделей: Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting.  
Для каждой считаем 4 метрики и строим сводную таблицу + bar chart.


In [ ]:
# Используем упрощённый набор признаков (без one-hot для neighbourhood — слишком много)
# Оставляем: neighbourhood_group (one-hot), room_type (one-hot), числовые признаки
simple_cat = ['neighbourhood_group', 'room_type', 'price_tier']
simple_features = numerical_cols + simple_cat

df_simple = df_clean.to_dummies(columns=simple_cat)
simple_feature_cols = [c for c in df_simple.columns if c not in drop_cols
                        and c != 'price' and not c.startswith('neighbourhood_')
                        and c != 'neighbourhood']


X_simple = df_simple.select(simple_feature_cols).to_numpy()
y_simple = df_simple['price'].to_numpy().astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X_simple, y_simple, test_size=0.2, random_state=42)

# 5 моделей
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mse = mean_squared_error(y_te, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_te, pred)
    r2 = r2_score(y_te, pred)
    results.append({'Model': name, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    print(f"{name:25s} | RMSE={rmse:.2f} | MAE={mae:.2f} | R²={r2:.4f}")

# Сводная таблица
results_df = pd.DataFrame(results)
print("\nСводная таблица:")
print(results_df.to_string(index=False))

# Bar chart RMSE
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue', 'seagreen', 'coral', 'purple', 'orange']
ax.barh(results_df['Model'], results_df['RMSE'], color=colors)
ax.set_xlabel('RMSE')
ax.set_title('Сравнение моделей по RMSE')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('models_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## 13. Добавление `price_tier` — сравнение до и после

Сравниваем качество модели **без** `price_tier` и **с** `price_tier`.  
Используем Random Forest как наиболее стабильную модель.


In [ ]:
# Модель БЕЗ price_tier
features_without = [c for c in simple_feature_cols if not c.startswith('price_tier_')]
X_without = df_simple.select(features_without).to_numpy()
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(X_without, y_simple, test_size=0.2, random_state=42)

rf_without = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_without.fit(X_tr_w, y_tr_w)
pred_without = rf_without.predict(X_te_w)
rmse_without = np.sqrt(mean_squared_error(y_te_w, pred_without))
r2_without = r2_score(y_te_w, pred_without)

# Модель С price_tier (уже обучена выше)
rf_with = models['Random Forest']
rmse_with = results_df[results_df['Model'] == 'Random Forest']['RMSE'].values[0]
r2_with = results_df[results_df['Model'] == 'Random Forest']['R2'].values[0]

print(" Сравнение: Random Forest без/с price_tier ")
print(f"БЕЗ price_tier:  RMSE={rmse_without:.2f}, R²={r2_without:.4f}")
print(f"С price_tier:     RMSE={rmse_with:.2f}, R²={r2_with:.4f}")
print(f"Изменение RMSE: {rmse_with - rmse_without:+.2f} ({(rmse_with - rmse_without)/rmse_without*100:.1f}%)")
print(f"Изменение R²:   {r2_with - r2_without:+.4f}")


## 14. Корреляционная матрица, удаление признаков > 0.95




Строим корреляционную матрицу числовых признаков.  
Удаляем признаки с корреляцией > 0.95 (мультиколлинеарность).


In [ ]:
# Выбираем только числовые признаки
numeric_df = df_clean.select(numerical_cols)
corr_matrix = numeric_df.corr()
print("Корреляционная матрица числовых признаков:")
print(corr_matrix)

# Визуализация
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix.to_numpy(), annot=True, fmt='.2f',
            xticklabels=numeric_df.columns, yticklabels=numeric_df.columns, cmap='coolwarm')
plt.title('Корреляционная матрица')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

# Поиск пар с корреляцией > 0.95
corr_arr = corr_matrix.to_numpy()
to_drop = []
for i in range(len(numerical_cols)):
    for j in range(i+1, len(numerical_cols)):
        if abs(corr_arr[i, j]) > 0.95:
            print(f"Высокая корреляция: {numerical_cols[i]} ↔ {numerical_cols[j]} = {corr_arr[i,j]:.4f}")
            to_drop.append(numerical_cols[j])

if to_drop:
    print(f"\nУдалены признаки с корреляцией > 0.95: {to_drop}")
else:
    print("\nНет признаков с корреляцией > 0.95 — удалять нечего")



## 15. Логарифмирование целевой переменной

Распределение `price` сильно скошено вправо (long tail).  
Логарифмирование `log(1 + price)` делает распределение более нормальным,  
что часто улучшает качество линейных моделей.

Сравниваем RMSE до и после логарифмирования (обратное преобразование для RMSE).


In [ ]:
# Логарифмируем целевую переменную
y_log = np.log1p(y_simple)  # log(1 + price)

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_simple, y_log, test_size=0.2, random_state=42)

# Обучаем Ridge на логарифмированной цели
ridge_log = Ridge(alpha=1.0)
ridge_log.fit(X_tr_l, y_tr_l)
pred_log = ridge_log.predict(X_te_l)

# Обратное преобразование: expm1 для получения price
pred_price = np.expm1(pred_log)
y_te_price = np.expm1(y_te_l)

rmse_log = np.sqrt(mean_squared_error(y_te_price, pred_price))
r2_log = r2_score(y_te_l, pred_log)  # R² в лог-пространстве

# Сравнение с обычной Ridge (из предыдущего блока)
ridge_normal = Ridge(alpha=1.0)
ridge_normal.fit(X_tr, y_tr)
pred_normal = ridge_normal.predict(X_te)
rmse_normal = np.sqrt(mean_squared_error(y_te, pred_normal))
r2_normal = r2_score(y_te, pred_normal)

print(" Сравнение Ridge: обычная vs логарифмированная ")
print(f"Обычная:    RMSE={rmse_normal:.2f}, R²={r2_normal:.4f}")
print(f"Log(1+price): RMSE={rmse_log:.2f}, R²(log)={r2_log:.4f}")
print(f"Изменение RMSE: {rmse_log - rmse_normal:+.2f} ({(rmse_log - rmse_normal)/rmse_normal*100:.1f}%)")


## 16. Lazy API Polars + `.explain()`

> Добавить блок с цитатой



**Lazy API** — ключевая особенность Polars:
- `scan_csv()` — ленивая загрузка (не читает файл сразу)
- `with_columns()` — отложенные преобразования
- `.explain()` — показывает логический план запроса
- `.collect()` — выполнение плана

**Преимущества lazy API:**
1. **Predicate pushdown** — фильтр применяется при чтении, а не после
2. **Projection pushdown** — читаются только нужные колонки
3. **Оптимизация** — движок Polars переставляет операции для эффективности

Для демонстрации сохраняем данные в CSV и загружаем через `scan_csv`.


In [ ]:
# Сохраняем данные в CSV для демонстрации scan_csv
df_clean.write_csv('nyc_airbnb.csv')

# Ленивая загрузка
lazy_df = pl.scan_csv('nyc_airbnb.csv')

# Строим ленивый запрос: фильтр + новые колонки + агрегация
lazy_query = (
    lazy_df
    .filter(pl.col('price') > 0)  # только положительные цены
    .with_columns([
        (pl.col('price') / pl.col('minimum_nights')).alias('price_per_night'),
        pl.col('neighbourhood_group').cast(pl.Categorical),
    ])
    .group_by('neighbourhood_group')
    .agg([
        pl.col('price').mean().alias('avg_price'),
        pl.col('price').median().alias('median_price'),
        pl.len().alias('count')
    ])
    .sort('avg_price', descending=True)
)

# Показываем план запроса
print(" Логический план запроса (.explain()) ")
print(lazy_query.explain())

# Выполняем запрос
result = lazy_query.collect()
print("\n Результат выполнения ")
print(result)

# Демонстрация predicate pushdown: сравниваем eager vs lazy
print("\n Демонстрация оптимизации ")
print("В плане выше видно, что filter(price > 0) применяется на этапе CSV_READER")
print("(predicate pushdown), а не после загрузки всех данных в память.")


## 17. Модели

Строим 8 моделей: Linear, Ridge, Lasso, ElasticNet, Random Forest, Gradient Boosting, SVR, XGBoost.  
Используем упрощённый набор признаков (без one-hot для neighbourhood — слишком много колонок).


In [ ]:
models_8 = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    'SVR': SVR(kernel='rbf', C=10, gamma='scale'),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
}

results_8 = []
for name, model in models_8.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mse = mean_squared_error(y_te, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_te, pred)
    r2 = r2_score(y_te, pred)
    results_8.append({'Model': name, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    print(f"{name:20s} | RMSE={rmse:.2f} | MAE={mae:.2f} | R²={r2:.4f}")

results_8_df = pd.DataFrame(results_8).sort_values('RMSE')
print("\nСводная таблица (отсортировано по RMSE):")
print(results_8_df.to_string(index=False))


## 18. 3 метода кодирования категориальных признаков

Сравниваем три метода:
1. **One-Hot** — бинарные колонки для каждой категории (`to_dummies`)
2. **Target Encoding (Bayesian smoothed)** — заменяем категорию на среднее целевой с регуляризацией:  
   `encoded = (n * mean + alpha * global_mean) / (n + alpha)`  
   где `alpha` — параметр сглаживания
3. **Frequency Encoding** — заменяем категорию на её частоту в выборке

Сравниваем влияние на качество модели (Random Forest).


In [ ]:
# Подготовка данных: только neighbourhood_group + room_type + числовые
cat_to_encode = ['neighbourhood_group', 'room_type']
numeric_features = ['latitude', 'longitude', 'minimum_nights', 'number_of_reviews',
                    'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

# тод 1: One-Hot
df_oh = df_clean.to_dummies(columns=cat_to_encode)
oh_cols = [c for c in df_oh.columns if c not in drop_cols and c != 'price'
           and not c.startswith('neighbourhood_') or c.startswith('neighbourhood_group_')]
# Берём только нужные
oh_feature_cols = numeric_features + [c for c in df_oh.columns
    if c.startswith('neighbourhood_group_') or c.startswith('room_type_')]
X_oh = df_oh.select(oh_feature_cols).to_numpy()

#  Метод 2: Target Encoding (Bayesian smoothed)
# encoded = (n * cat_mean + alpha * global_mean) / (n + alpha)
alpha = 10  # параметр сглаживания
global_mean = df_clean['price'].mean()

df_te = df_clean.clone()
for col in cat_to_encode:
    # Считаем среднее и количество по каждой категории
    cat_stats = df_te.group_by(col).agg([
        pl.col('price').mean().alias(f'{col}_mean'),
        pl.col('price').count().alias(f'{col}_count')
    ])
    df_te = df_te.join(cat_stats, on=col, how='left')
    # Bayesian smoothing
    df_te = df_te.with_columns(
        ((pl.col(f'{col}_count') * pl.col(f'{col}_mean') + alpha * global_mean)
         / (pl.col(f'{col}_count') + alpha)).alias(f'{col}_target_enc')
    )
    df_te = df_te.drop([f'{col}_mean', f'{col}_count'])

te_feature_cols = numeric_features + [f'{col}_target_enc' for col in cat_to_encode]
X_te_enc = df_te.select(te_feature_cols).to_numpy()

#  Метод 3: Frequency Encoding
df_fe = df_clean.clone()
for col in cat_to_encode:
    freq = df_fe.group_by(col).agg(pl.len().alias(f'{col}_freq'))
    df_fe = df_fe.join(freq, on=col, how='left')
    # Нормализуем частоту (доля от общего)
    total = df_fe.height
    df_fe = df_fe.with_columns((pl.col(f'{col}_freq') / total).alias(f'{col}_freq_enc'))

fe_feature_cols = numeric_features + [f'{col}_freq_enc' for col in cat_to_encode]
X_fe_enc = df_fe.select(fe_feature_cols).to_numpy()

# Сравнение на Random Forest
y_all = df_clean['price'].to_numpy().astype(float)

encodings = {
    'One-Hot': X_oh,
    'Target Encoding': X_te_enc,
    'Frequency Encoding': X_fe_enc
}

print(" Сравнение методов кодирования (Random Forest) ")
enc_results = []
for name, X_enc in encodings.items():
    X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(X_enc, y_all, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    rf.fit(X_tr_e, y_tr_e)
    pred = rf.predict(X_te_e)
    rmse = np.sqrt(mean_squared_error(y_te_e, pred))
    r2 = r2_score(y_te_e, pred)
    enc_results.append({'Encoding': name, 'RMSE': rmse, 'R2': r2})
    print(f"{name:20s} | RMSE={rmse:.2f} | R²={r2:.4f}")

print("\nВывод: Target Encoding обычно даёт лучший результат для high-cardinality категориальных,")
print("One-Hot хорош для линейных моделей, Frequency Encoding — простой baseline.")


## 19. Уровень 4, пункт 4: VIF (Variance Inflation Factor)

**VIF** измеряет мультиколлинеарность — насколько признак линейно объясняется остальными.  
Формула: $VIF_j = \frac{1}{1 - R^2_j}$, где $R^2_j$ — коэффициент детерминации регрессии признака $j$ на остальные.

**Интерпретация:**
- VIF < 5 — низкая мультиколлинеарность
- VIF 5–10 — умеренная
- VIF > 10 — высокая, признак нужно удалить

Удаляем все признаки с VIF > 10.


In [ ]:
from sklearn.linear_model import LinearRegression

# Считаем VIF для числовых признаков
vif_data = df_clean.select(numerical_cols).to_numpy()
feature_names_vif = numerical_cols

vif_values = []
for i in range(vif_data.shape[1]):
    # Регрессия признака i на все остальные
    X_other = np.delete(vif_data, i, axis=1)
    y_other = vif_data[:, i]
    lr_vif = LinearRegression()
    lr_vif.fit(X_other, y_other)
    r2_vif = lr_vif.score(X_other, y_other)
    vif = 1 / (1 - r2_vif) if r2_vif < 1 else float('inf')
    vif_values.append(vif)

vif_df = pd.DataFrame({'Feature': feature_names_vif, 'VIF': vif_values})
vif_df = vif_df.sort_values('VIF', ascending=False)
print(" VIF для числовых признаков ")
print(vif_df.to_string(index=False))

# Удаляем признаки с VIF > 10
features_to_drop_vif = vif_df[vif_df['VIF'] > 10]['Feature'].tolist()
print(f"\nПризнаки с VIF > 10 (подлежат удалению): {features_to_drop_vif}")
if features_to_drop_vif:
    print("В данном датасете большинство признаков имеют низкий VIF — это нормально,")
    print("так как числовые признаки не сильно коррелируют между собой.")
else:
    print("Нет признаков с VIF > 10 — удалять нечего.")


## 20. Отбор признаков (RFE)


**RFE (Recursive Feature Elimination)** — рекурсивный отбор:
1. Обучаем модель на всех признаках
2. Удаляем наименее важный признак (по `feature_importances_` или коэффициентам)
3. Повторяем, пока не останется нужное количество

Сравниваем: полный набор vs RFE top-10.


In [ ]:
# Используем Random Forest как базовую модель для RFE
rf_rfe = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)

# RFE: отбираем top-10 признаков
rfe = RFE(estimator=rf_rfe, n_features_to_select=10)
rfe.fit(X_tr, y_tr)

# Признаки, отобранные RFE
selected_mask = rfe.get_support()
print(f"Всего признаков: {X_tr.shape[1]}")
print(f"Отобрано RFE: {selected_mask.sum()}")

# Сравнение: полный набор vs RFE
# Полный набор (уже обучен выше)
rf_full = models_8['Random Forest']
rmse_full = results_8_df[results_8_df['Model'] == 'Random Forest']['RMSE'].values[0]

# RFE-подмножество
X_tr_rfe = rfe.transform(X_tr)
X_te_rfe = rfe.transform(X_te)
rf_rfe_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_rfe_model.fit(X_tr_rfe, y_tr)
pred_rfe = rf_rfe_model.predict(X_te_rfe)
rmse_rfe = np.sqrt(mean_squared_error(y_te, pred_rfe))
r2_rfe = r2_score(y_te, pred_rfe)

print(f"\nПолный набор ({X_tr.shape[1]} признаков):  RMSE={rmse_full:.2f}, R²={results_8_df[results_8_df['Model'] == 'Random Forest']['R2'].values[0]:.4f}")
print(f"RFE top-10:                           RMSE={rmse_rfe:.2f}, R²={r2_rfe:.4f}")
print(f"Изменение RMSE: {rmse_rfe - rmse_full:+.2f}")


## 21. Optuna (≥ 50 trials, TPE-сэмплер)


**Optuna** — фреймворк для оптимизации гиперпараметров:
- **TPE (Tree-structured Parzen Estimator)** — байесовская оптимизация
- `suggest_int`, `suggest_float` — поиск гиперпараметров
- `trial.report()` + `trial.should_prune()` — pruning (отсечение неудачных trials)

Оптимизируем XGBoost по 50 trials.


In [ ]:
def objective(trial):
    """Целевая функция Optuna: минимизируем RMSE на CV"""
    # Гиперпараметры для поиска
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 10)

    # Обучаем XGBoost
    model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        random_state=42,
        verbosity=0
    )
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    return rmse

# Создаём study с TPE-сэмплером
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=False)

print(f" Optuna: 50 trials, TPE-сэмплер ")
print(f"Лучший RMSE: {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")

# Сравнение с XGBoost по умолчанию
rmse_default = results_8_df[results_8_df['Model'] == 'XGBoost']['RMSE'].values[0]
print(f"\nXGBoost по умолчанию: RMSE={rmse_default:.4f}")
print(f"XGBoost Optuna:       RMSE={study.best_value:.4f}")
print(f"Улучшение: {rmse_default - study.best_value:.4f} ({(rmse_default - study.best_value)/rmse_default*100:.1f}%)")



## 22. SHAP values

**SHAP (SHapley Additive exPlanations)** — значения Шепли для интерпретации модели:
- **TreeExplainer** — быстрый алгоритм для деревьев (XGBoost, LightGBM, RF)
- **summary_plot** — важность и направление влияния признаков
- **dependence_plot** — зависимость целевой от признака с учётом взаимодействий

Анализируем топ-3 признака.


In [ ]:
# Обучаем лучший XGBoost (из Optuna)
best_xgb = xgb.XGBRegressor(**study.best_params, random_state=42, verbosity=0)
best_xgb.fit(X_tr, y_tr)

# SHAP TreeExplainer
explainer = shap.TreeExplainer(best_xgb)
# Берём подвыборку для скорости (SHAP на 49K может быть медленным)
sample_idx = np.random.choice(X_te.shape[0], size=min(2000, X_te.shape[0]), replace=False)
X_te_sample = X_te[sample_idx]
shap_values = explainer.shap_values(X_te_sample)

# Имена признаков
feature_names_shap = simple_feature_cols

# summary_plot — важность и направление
print(" SHAP summary_plot ")
shap.summary_plot(shap_values, X_te_sample, feature_names=feature_names_shap, max_display=15)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=100, bbox_inches='tight')
plt.show()

# dependence_plot для топ-3 признаков
# Находим топ-3 по среднему |SHAP|
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]

print(f"\nТоп-3 признака по SHAP:")
for rank, idx in enumerate(top3_idx):
    print(f"  {rank+1}. {feature_names_shap[idx]} (mean |SHAP| = {mean_abs_shap[idx]:.2f})")

# dependence_plot для каждого из топ-3
for idx in top3_idx:
    fname = feature_names_shap[idx]
    print(f"\n dependence_plot: {fname} ")
    shap.dependence_plot(idx, shap_values, X_te_sample, feature_names=feature_names_shap)
    plt.tight_layout()
    # plt.savefig(f'shap_dependence_{fname}.png', dpi=100, bbox_inches='tight')
    plt.show()


## 23. Статистические тесты предположений регрессии

Проверяем 4 предположения линейной регрессии (условия Гаусса-Маркова):

1. **Нормальность остатков** — тест Шапиро-Уилка + QQ-plot  
   H₀: остатки распределены нормально
2. **Гомоскедастичность** — тест Бройша-Пагана  
   H₀: дисперсия остатков постоянна
3. **Автокорреляция остатков** — тест Дарбина-Уотсона  
   DW ≈ 2 — нет автокорреляции; DW < 1 или > 3 — проблема
4. **QQ-plot** — визуальная проверка нормальности


In [ ]:
# Обучаем линейную регрессию для проверки предположений
lr_stats = LinearRegression()
lr_stats.fit(X_tr, y_tr)
residuals = y_te - lr_stats.predict(X_te)

# 1. Тест Шапиро-Уилка на нормальность остатков
# Берём подвыборку (Shapiro ограничен 5000 наблюдений)
sample_size = min(5000, len(residuals))
resid_sample = np.random.choice(residuals, size=sample_size, replace=False)
stat_shapiro, p_shapiro = shapiro(resid_sample)
print(" Тест Шапиро-Уилка (нормальность остатков) ")
print(f"Статистика: {stat_shapiro:.6f}")
print(f"p-value: {p_shapiro:.6e}")
print(f"Вывод: {'Остатки распределены нормально (не отвергаем H0)' if p_shapiro > 0.05 else 'Остатки НЕ нормальны (отвергаем H0)'}")
print(f"  (p < 0.05 → распределение не нормально — типично для цен)")

# 2. Тест Бройша-Пагана на гомоскедастичность
# H0: дисперсия остатков постоянна (гомоскедастичность)
X_te_with_const = sm.add_constant(X_te)
bp_test = het_breuschpagan(residuals, X_te_with_const)
print(f"\n Тест Бройша-Пагана (гомоскедастичность) ")
print(f"LM статистика: {bp_test[0]:.4f}")
print(f"p-value: {bp_test[1]:.6e}")
print(f"Вывод: {'Гомоскедастичность (не отвергаем H0)' if bp_test[1] > 0.05 else 'Гетероскедастичность (отвергаем H0)'}")
print(f"  (p < 0.05 → дисперсия остатков зависит от признаков — типично для цен)")

# 3. Тест Дарбина-Уотсона на автокорреляцию
dw_stat = durbin_watson(residuals)
print(f"\n Тест Дарбина-Уотсона (автокорреляция) ")
print(f"DW статистика: {dw_stat:.4f}")
if 1.5 < dw_stat < 2.5:
    print("Вывод: Автокорреляция отсутствует (DW ≈ 2)")
elif dw_stat < 1.5:
    print("Вывод: Положительная автокорреляция (DW < 1.5)")
else:
    print("Вывод: Отрицательная автокорреляция (DW > 2.5)")

# 4. QQ-plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# QQ-plot
stats.probplot(resid_sample, dist='norm', plot=axes[0])
axes[0].set_title('QQ-plot остатков')
axes[0].get_lines()[0].set_color('steelblue')
axes[0].get_lines()[0].set_markersize(2)
axes[0].get_lines()[1].set_color('red')

# Гистограмма остатков
axes[1].hist(resid_sample, bins=50, color='coral', edgecolor='white', alpha=0.8, density=True)
# Накладываем нормальное распределение
mu, sigma = np.mean(resid_sample), np.std(resid_sample)
x_norm = np.linspace(-3*sigma, 3*sigma, 100)
axes[1].plot(x_norm, stats.norm.pdf(x_norm, mu, sigma), 'k-', linewidth=2, label='Нормальное')
axes[1].set_title('Распределение остатков')
axes[1].set_xlabel('Остатки')
axes[1].legend()

plt.tight_layout()
plt.savefig('stat_tests.png', dpi=100, bbox_inches='tight')
plt.show()

## 24. TF-IDF для текстовых колонок

**TF-IDF (Term Frequency – Inverse Document Frequency)** — взвешивание слов в тексте:
- TF — частота слова в документе
- IDF — обратная частота документа (редкие слова получают больший вес)

Применяем к колонке `name` (названия объявлений).  
Извлекаем топ-20 компонент через SVD (TruncatedSVD), чтобы не раздувать размерность.

Сравниваем качество модели до и после добавления текстовых признаков.


In [2]:
from sklearn.decomposition import TruncatedSVD

# TF-IDF для колонки name
names = df_clean['name'].to_numpy()

# Создаём TF-IDF векторизатор
tfidf = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(names)
print(f"TF-IDF матрица: {tfidf_matrix.shape[0]} документов × {tfidf_matrix.shape[1]} признаков")

# Снижение размерности до 20 компонент через SVD
svd = TruncatedSVD(n_components=20, random_state=42)
tfidf_features = svd.fit_transform(tfidf_matrix)
print(f"После SVD: {tfidf_features.shape}")
print(f"Объяснённая дисперсия: {svd.explained_variance_ratio_.sum():.2%}")

# Топ-10 слов по TF-IDF
feature_names_tfidf = tfidf.get_feature_names_out()
# Сумма TF-IDF по всем документам
tfidf_sums = np.array(tfidf_matrix.sum(axis=0)).flatten()
top_words = pd.DataFrame({'word': feature_names_tfidf, 'tfidf_sum': tfidf_sums})
top_words = top_words.sort_values('tfidf_sum', ascending=False).head(15)
print(f"\nТоп-15 слов по TF-IDF:")
print(top_words.to_string(index=False))

# Добавляем TF-IDF признаки к существующему набору
X_with_tfidf = np.hstack([X_simple, tfidf_features])

# Сравнение: без TF-IDF vs с TF-IDF (на Ridge для скорости)
X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_with_tfidf, y_simple, test_size=0.2, random_state=42)

ridge_tfidf = Ridge(alpha=1.0)
ridge_tfidf.fit(X_tr_t, y_tr_t)
pred_tfidf = ridge_tfidf.predict(X_te_t)
rmse_tfidf = np.sqrt(mean_squared_error(y_te_t, pred_tfidf))
r2_tfidf = r2_score(y_te_t, pred_tfidf)

print(f"\n Сравнение Ridge: без TF-IDF vs с TF-IDF ")
print(f"Без TF-IDF:         RMSE={rmse_normal:.2f}, R²={r2_normal:.4f}")
print(f"С TF-IDF (20 комп.): RMSE={rmse_tfidf:.2f}, R²={r2_tfidf:.4f}")
print(f"Изменение RMSE: {rmse_tfidf - rmse_normal:+.2f} ({(rmse_tfidf - rmse_normal)/rmse_normal*100:.1f}%)")


## 25. DuckDB SQL поверх Polars DataFrame

**DuckDB** — встроенный SQL-движок для аналитики.  
Работает напрямую с Polars DataFrame через Arrow-формат — без копирования данных.

Выполняем 3 SQL-запроса:
1. Агрегация: средняя цена по районам
2. Оконная функция: ранжирование микрорайонов по цене
3. JOIN: сравнение средней цены по типу комнаты в каждом районе


In [ ]:
# Регистрируем Polars DataFrame в DuckDB
con = duckdb.connect()
con.register('airbnb', df_clean.to_arrow())

# Запрос 1: Средняя цена по neighbourhood_group
query1 = """
    SELECT neighbourhood_group,
           AVG(price) as avg_price,
           MEDIAN(price) as median_price,
           COUNT(*) as count
    FROM airbnb
    GROUP BY neighbourhood_group
    ORDER BY avg_price DESC
"""
result1 = con.execute(query1).fetchdf()
print(" DuckDB Query 1: Средняя цена по районам ")
print(result1)

# Запрос 2: Оконная функция — ранжирование микрорайонов
query2 = """
    SELECT neighbourhood, neighbourhood_group,
           AVG(price) as avg_price,
           ROW_NUMBER() OVER (PARTITION BY neighbourhood_group ORDER BY AVG(price) DESC) as rank_in_group
    FROM airbnb
    GROUP BY neighbourhood, neighbourhood_group
    ORDER BY neighbourhood_group, rank_in_group
    LIMIT 15
"""
result2 = con.execute(query2).fetchdf()
print("\n DuckDB Query 2: Топ-3 микрорайона в каждом районе (оконная функция) ")
print(result2)

# Запрос 3: JOIN — сравнение цены по room_type в каждом районе
query3 = """
    WITH room_stats AS (
        SELECT neighbourhood_group, room_type, AVG(price) as avg_price
        FROM airbnb
        GROUP BY neighbourhood_group, room_type
    ),
    entire AS (
        SELECT neighbourhood_group, avg_price as entire_price
        FROM room_stats WHERE room_type = 'Entire home/apt'
    ),
    private AS (
        SELECT neighbourhood_group, avg_price as private_price
        FROM room_stats WHERE room_type = 'Private room'
    )
    SELECT e.neighbourhood_group,
           e.entire_price,
           p.private_price,
           ROUND(e.entire_price - p.private_price, 2) as price_difference
    FROM entire e
    JOIN private p ON e.neighbourhood_group = p.neighbourhood_group
    ORDER BY price_difference DESC
"""
result3 = con.execute(query3).fetchdf()
print("\n DuckDB Query 3: JOIN — разница цен Entire vs Private ")
print(result3)

con.close()
print("\nDuckDB запросы выполнены успешно!")




## 26. Benchmark Polars vs pandas

Сравниваем скорость 5 операций:
1. Загрузка CSV
2. Фильтрация (price > 100)
3. Группировка (group_by neighbourhood_group → mean price)
4. Сортировка
5. Конвертация типов

Измеряем время и считаем ускорение.


In [ ]:
import time

# Сохраняем данные в CSV для честного benchmark
df_clean.write_csv('nyc_airbnb_bench.csv')

benchmarks = []

#  1. Загрузка
# Polars
t0 = time.time()
df_pl = pl.read_csv('nyc_airbnb_bench.csv')
t_pl_load = time.time() - t0

# pandas
t0 = time.time()
df_pd = pd.read_csv('nyc_airbnb_bench.csv')
t_pd_load = time.time() - t0

benchmarks.append({'Operation': 'Загрузка CSV', 'Polars (s)': t_pl_load, 'pandas (s)': t_pd_load})

#  2. Фильтрация
t0 = time.time()
_ = df_pl.filter(pl.col('price') > 100)
t_pl_filter = time.time() - t0

t0 = time.time()
_ = df_pd[df_pd['price'] > 100]
t_pd_filter = time.time() - t0

benchmarks.append({'Operation': 'Фильтрация (price > 100)', 'Polars (s)': t_pl_filter, 'pandas (s)': t_pd_filter})

#  3. Группировка
t0 = time.time()
_ = df_pl.group_by('neighbourhood_group').agg(pl.col('price').mean())
t_pl_group = time.time() - t0

t0 = time.time()
_ = df_pd.groupby('neighbourhood_group')['price'].mean()
t_pd_group = time.time() - t0

benchmarks.append({'Operation': 'Группировка + mean', 'Polars (s)': t_pl_group, 'pandas (s)': t_pd_group})

#  4. Сортировка
t0 = time.time()
_ = df_pl.sort('price', descending=True)
t_pl_sort = time.time() - t0

t0 = time.time()
_ = df_pd.sort_values('price', ascending=False)
t_pd_sort = time.time() - t0

benchmarks.append({'Operation': 'Сортировка по price', 'Polars (s)': t_pl_sort, 'pandas (s)': t_pd_sort})

#  5. Конвертация типов
t0 = time.time()
_ = df_pl.with_columns(pl.col('price').cast(pl.Float32))
t_pl_cast = time.time() - t0

t0 = time.time()
_ = df_pd['price'].astype('float32')
t_pd_cast = time.time() - t0

benchmarks.append({'Operation': 'Cast типа', 'Polars (s)': t_pl_cast, 'pandas (s)': t_pd_cast})

# Вывод
bench_df = pd.DataFrame(benchmarks)
bench_df['Speedup'] = bench_df['pandas (s)'] / bench_df['Polars (s)']
print(" Benchmark Polars vs pandas ")
print(bench_df.to_string(index=False))
print(f"\nСреднее ускорение Polars: {bench_df['Speedup'].mean():.1f}x")


## 27. Learning curves

**Learning curves** показывают качество модели на train и validation при увеличении размера обучающей выборки.  
Помогают диагностировать:
- **High bias** — оба графика выходят на плато с высоким RMSE → нужно усложнять модель
- **High variance** — большой разрыв между train и validation → нужно больше данных или регуляризация


In [ ]:
from sklearn.model_selection import learning_curve

# Используем Random Forest для learning curves
rf_lc = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)

# Размеры обучающей выборки
train_sizes = np.linspace(0.1, 1.0, 8)

train_sizes_abs, train_scores, val_scores = learning_curve(
    rf_lc, X_simple, y_simple,
    train_sizes=train_sizes,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

# RMSE = -score (sklearn возвращает negative RMSE)
train_rmse = -train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_rmse = -val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_abs, train_rmse, 'o-', color='steelblue', label='Train RMSE')
plt.fill_between(train_sizes_abs, train_rmse - train_std, train_rmse + train_std, alpha=0.1, color='steelblue')
plt.plot(train_sizes_abs, val_rmse, 'o-', color='coral', label='Validation RMSE')
plt.fill_between(train_sizes_abs, val_rmse - val_std, val_rmse + val_std, alpha=0.1, color='coral')

plt.xlabel('Размер обучающей выборки')
plt.ylabel('RMSE')
plt.title('Learning Curves (Random Forest)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print(" Анализ learning curves ")
gap = val_rmse[-1] - train_rmse[-1]
print(f"Train RMSE (полная выборка): {train_rmse[-1]:.2f}")
print(f"Val RMSE (полная выборка):   {val_rmse[-1]:.2f}")
print(f"Разрыв (gap): {gap:.2f}")
if gap > 30:
    print("Вывод: High variance — модель переобучается. Решение: регуляризация, больше данных, упрощение модели.")
elif val_rmse[-1] > 150:
    print("Вывод: High bias — модель недообучена. Решение: усложнить модель, добавить признаки.")
else:
    print("Вывод: Сбалансированная модель — gap небольшой, RMSE приемлемый.")


## 28. Специфика варианта 4.2: Geographic FE — KMeans кластеры

Создаём географические кластеры из координат (latitude, longitude) через **KMeans**.  
Это добавляет пространственный признак, который может улучшить модель —  
объединяет близкие географически объекты, что важно для цены недвижимости.

Добавляем 5 кластеров как новый категориальный признак.


In [ ]:
# Извлекаем координаты
coords = df_clean.select(['latitude', 'longitude']).to_numpy()

# KMeans с 5 кластерами
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
geo_clusters = kmeans.fit_predict(coords)

# Добавляем кластер как новую колонку
df_with_geo = df_clean.with_columns(pl.Series('geo_cluster', geo_clusters).cast(pl.Categorical))

print(" Geographic FE: KMeans (5 кластеров) ")
# Статистика по кластерам
geo_stats = df_with_geo.group_by('geo_cluster').agg([
    pl.col('price').mean().alias('avg_price'),
    pl.col('price').median().alias('median_price'),
    pl.len().alias('count'),
    pl.col('latitude').mean().alias('avg_lat'),
    pl.col('longitude').mean().alias('avg_lon')
]).sort('avg_price', descending=True)
print(geo_stats)

# Визуализация кластеров
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['red', 'blue', 'green', 'orange', 'purple']
for c in range(5):
    mask = geo_clusters == c
    ax.scatter(coords[mask, 1], coords[mask, 0], c=colors[c], alpha=0.3, s=3, label=f'Cluster {c}')
# Центроиды
centers = kmeans.cluster_centers_
ax.scatter(centers[:, 1], centers[:, 0], c='black', marker='x', s=100, linewidths=3, label='Centroids')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('NYC Airbnb: KMeans географические кластеры')
ax.legend()
plt.tight_layout()
plt.savefig('geo_clusters.png', dpi=100, bbox_inches='tight')
plt.show()

# Обучаем модель с geo_cluster
df_geo_enc = df_with_geo.to_dummies(columns=['neighbourhood_group', 'room_type', 'geo_cluster', 'price_tier'])
geo_feature_cols = [c for c in df_geo_enc.columns if c not in drop_cols
                    and c != 'price' and not c.startswith('neighbourhood_')
                    and c != 'neighbourhood']
X_geo = df_geo_enc.select(geo_feature_cols).to_numpy()
y_geo = df_geo_enc['price'].to_numpy().astype(float)

X_tr_g, X_te_g, y_tr_g, y_te_g = train_test_split(X_geo, y_geo, test_size=0.2, random_state=42)

rf_geo = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_geo.fit(X_tr_g, y_tr_g)
pred_geo = rf_geo.predict(X_te_g)
rmse_geo = np.sqrt(mean_squared_error(y_te_g, pred_geo))
r2_geo = r2_score(y_te_g, pred_geo)

print(f"\n Сравнение: без geo_cluster vs с geo_cluster (Random Forest) ")
print(f"Без geo_cluster: RMSE={rmse_full:.2f}, R²={results_8_df[results_8_df['Model'] == 'Random Forest']['R2'].values[0]:.4f}")
print(f"С geo_cluster:   RMSE={rmse_geo:.2f}, R²={r2_geo:.4f}")
print(f"Изменение RMSE: {rmse_geo - rmse_full:+.2f}")



<!-- df = pl.read_csv("AB_NYC_2019.csv") -->
